# Milestone 0: StockCortex Architecture, Hardware Calibration & Setup

## 1. What We Built
- Established the `stockcortex` repository structure separating immutable Medallion layers (`data/bronze`, `data/silver`, `data/gold`), core Python packages (`src/stockcortex/`), automated `pytest` assertions (`tests/`), and walkthrough notebooks (`notebooks/`).
- Implemented `src/stockcortex/config.py` using Pydantic `BaseSettings` to manage environment keys, filesystem paths, 16 GB RAM hardware budgets, and the curated NIFTY starter equity universe.

## 2. Why These Architectural Decisions Were Made
| Engineering Choice | Rationale & Trade-Off |
| :--- | :--- |
| **DuckDB `memory_limit='6GB'`, `threads=4`** | Caps analytical query RAM on a 16 GB Windows 11 machine so DuckDB never causes OS paging while leaving headroom for PyTorch FinBERT (~1.2 GB RSS) and LightGBM. |
| **Pure `httpx` + Pydantic v2 (No `openai` SDK)** | Avoids Windows Smart App Control blocking the unsigned `jiter` Rust binary while enforcing strict JSON schema validation. |
| **Medallion Storage (`Bronze -> Silver -> Gold`)** | Isolates raw API snapshots from split-adjusted bars and strictly causal ML features to eliminate accidental mutation or look-ahead leakage. |

## 3. How to Verify Configuration & Environment

In [1]:
from stockcortex.config import STARTER_UNIVERSE, settings

print(f"Project Root        : {settings.base_dir}")
print(f"SQLite Ledger Path  : {settings.sqlite_db_path}")
print(f"DuckDB Memory Cap   : {settings.duckdb_memory_limit} ({settings.duckdb_threads} threads)")
print(f"FinBERT CPU Config  : batch_size={settings.finbert_batch_size}, max_length={settings.finbert_max_length}")
print(f"Starter Universe ({len(STARTER_UNIVERSE)}): {list(STARTER_UNIVERSE.keys())}")

Project Root        : C:\Users\jaich\Documents\Projects\stockcortex
SQLite Ledger Path  : C:\Users\jaich\Documents\Projects\stockcortex\data\stockcortex.db
DuckDB Memory Cap   : 6GB (4 threads)
FinBERT CPU Config  : batch_size=16, max_length=128
Starter Universe (10): ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK', 'BHARTIARTL', 'SBIN', 'LT', 'ITC', 'HINDUNILVR']
